# Week 1: Introduction, the Optimization Workflow, and the Tool Stack

**Course:** 2105623 Optimization of Chemical Processes  
**Institution:** Department of Chemical Engineering, Chulalongkorn University  
**Instructor:** Assoc. Prof. Dr. Soorathep Kheawhom

**Week:** 1 of 15 (modeling block, Weeks 1 to 6)  
**CLO mapping:** CLO 1 (formulate models: objective, decision variables, constraints), CLO 4 (implement and solve with computational tools such as Pyomo)

## Learning objectives

By the end of this notebook you should be able to:

- Name the four parts of any optimization problem (decision variables, parameters, objective, constraints) and identify each one in a written problem description.
- State the five stages of the optimization workflow (verbal problem, structured specification, algebraic model, code, solution and interpretation), name the artifact each stage produces, and name the characteristic failure of each stage.
- Describe what the modeling layer (Excel Solver, OpenSolver, Pyomo) and the solver layer each do, and select the solver class (LP, MILP, NLP, MINLP) that a stated problem requires.
- Write a small product-mix problem algebraically, translate every algebraic object into the matching Pyomo component, and confirm that a graphical solution, a spreadsheet layout and Pyomo return the same answer.
- Classify a problem as LP, NLP, IP, MILP, stochastic or global from its written description, and recognize infeasibility and unboundedness from the solver termination condition.

**Estimated duration:** 120 minutes  
**Prerequisites:** undergraduate calculus and linear algebra, basic Python. No prior optimization course is assumed.

**Reference:** Rao, *Engineering Optimization: Theory and Practice*, 4th ed., Ch. 1; Edgar, Himmelblau and Lasdon, *Optimization of Chemical Processes*, Ch. 1 and Ch. 7; Williams, *Model Building in Mathematical Programming*, Ch. 1.

This is the first notebook of the course, so it moves slowly and repeats itself on purpose. Later notebooks assume everything below. Sections 2 and 3 set up the two things that the whole modeling block (Weeks 1 to 6) reuses: the workflow that turns a sentence into a solved model, and the software stack that executes it.


In [ ]:
# --- Environment check -------------------------------------------------------
import sys, subprocess, importlib, shutil

def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

for p, n in [("pyomo", "pyomo"), ("numpy", "numpy"), ("scipy", "scipy"),
             ("matplotlib", "matplotlib"), ("pandas", "pandas")]:
    ensure(p, n)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pyomo.environ as pyo

def pick_solver(kind="lp"):
    """Return the first available solver of the requested kind."""
    order = {"lp":   ["appsi_highs", "glpk", "cbc", "gurobi", "cplex"],
             "milp": ["appsi_highs", "cbc", "glpk", "gurobi", "cplex"],
             "nlp":  ["ipopt", "conopt", "knitro"],
             "minlp":["bonmin", "couenne", "mindtpy"]}[kind]
    for name in order:
        try:
            s = pyo.SolverFactory(name)
            if s is not None and s.available(exception_flag=False):
                print(f"Using solver: {name}")
                return s
        except Exception:
            continue
    raise RuntimeError(f"No {kind} solver found. Install one, e.g. 'pip install highspy' "
                       f"or 'conda install -c conda-forge ipopt glpk coincbc'.")

In [ ]:
# --- Figure style: Teal-Amber Lab Palette v1.0 -------------------------------
PALETTE = ["#0F6E6B", "#E29A2D", "#BE654C", "#5A91BE", "#83A462", "#995A90", "#333F4A", "#DFC98F"]
INK, GRAPHITE, MIST, PAPER = "#1C242B", "#333F4A", "#B9C1C6", "#F3F0EB"

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRAPHITE, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.linewidth": 1.0, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": MIST, "grid.linewidth": 0.7, "grid.alpha": 0.9,
    "xtick.color": GRAPHITE, "ytick.color": GRAPHITE,
    "text.color": INK, "lines.linewidth": 1.8, "lines.markersize": 5,
    "font.size": 9, "legend.frameon": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

def tidy(ax):
    """Apply the house style to a single Axes object."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRAPHITE)
    return ax

print("Palette loaded:", ", ".join(PALETTE[:3]), "...")

## 1. What an optimization problem is

An optimization problem has exactly four parts. Everything else is detail.

1. **Decision variables.** The quantities you are free to choose. Collect them in the decision vector `x` with `n` components. A quantity that you cannot change is not a variable, it is a parameter.
2. **Parameters.** Numbers fixed before the problem is solved: prices, capacities, yields, demands, physical constants. They are the data of the problem.
3. **Objective function.** One scalar function `f(x)` that measures how good a choice of `x` is. Maximizing `f` is the same as minimizing `-f`, so nothing is lost by writing every problem as a minimization.
4. **Constraints.** The rules that a choice must satisfy. Inequality constraints are written `g(x) <= 0` and equality constraints `h(x) = 0`. Simple bounds such as `x >= 0` are constraints too, but solvers treat them separately because they are cheap to enforce.

The standard form used throughout this course is

    minimize    f(x)
    subject to  g_j(x) <= 0        j = 1, ..., m
                h_k(x) = 0         k = 1, ..., p
                x_L <= x <= x_U

A vector `x` that satisfies every constraint is **feasible**. The set of all feasible vectors is the **feasible region**. A feasible `x*` with `f(x*) <= f(x)` for every feasible `x` is a **global minimizer**, and `f(x*)` is the **optimal value**.

Two warnings that will recur all semester. First, the model is not the plant: it is a deliberate simplification, and the quality of the answer is limited by the quality of that simplification. Second, an optimizer will exploit every error in the model, so a model that is wrong in a way that pays will return an answer that is confidently wrong.

### Algebra to Pyomo, side by side

Pyomo is an algebraic modeling language. Its whole design is that each algebraic object above has one named component. The table is the map you will use for the rest of the course.

| Algebraic object | Mathematical notation | Pyomo component | Typical code |
|---|---|---|---|
| The model itself | the whole problem | `ConcreteModel` | `m = pyo.ConcreteModel()` |
| Index set | `i in I` | `Set` | `m.I = pyo.Set(initialize=["A", "B"])` |
| Parameter | `c_i` | `Param` | `m.c = pyo.Param(m.I, initialize=margin)` |
| Decision variable | `x_i` | `Var` | `m.x = pyo.Var(m.I, domain=pyo.NonNegativeReals)` |
| Simple bound | `0 <= x_i <= u_i` | `Var(bounds=...)` or `domain` | `pyo.Var(bounds=(0, 10))` |
| Objective | `min f(x)` or `max f(x)` | `Objective` | `m.profit = pyo.Objective(rule=..., sense=pyo.maximize)` |
| Inequality constraint | `g_j(x) <= 0` | `Constraint` | `m.res = pyo.Constraint(m.R, rule=...)` |
| Equality constraint | `h_k(x) = 0` | `Constraint` | `expr = lhs == rhs` |
| Integrality | `x_i` integer | `domain=NonNegativeIntegers` | `pyo.Var(domain=pyo.Binary)` |
| Algorithm | simplex, interior point, ... | `SolverFactory` | `pyo.SolverFactory("glpk")` |
| Optimal value | `f(x*)` | `value()` | `pyo.value(m.profit)` |
| Multiplier, shadow price | `lambda`, `mu` | `Suffix` | `m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)` |

The point of the table is that there is no translation step to invent. If you can write the algebra, the code is mechanical.

In [ ]:
# --- The four parts, made concrete in a two-line model -----------------------
# A deliberately trivial model, built only to show what each component looks like.
demo = pyo.ConcreteModel(name="anatomy_demo")

demo.I = pyo.Set(initialize=["A", "B"])                       # index set
demo.c = pyo.Param(demo.I, initialize={"A": 400.0, "B": 300.0})  # parameters
demo.x = pyo.Var(demo.I, domain=pyo.NonNegativeReals)         # decision variables
demo.cap = pyo.Constraint(expr=sum(demo.x[i] for i in demo.I) <= 9.0)   # constraint
demo.margin = pyo.Objective(expr=sum(demo.c[i] * demo.x[i] for i in demo.I),
                            sense=pyo.maximize)               # objective

mapping = pd.DataFrame(
    [["model",                "the whole problem",        "ConcreteModel", type(demo).__name__],
     ["index set",            "i in I",                   "Set",           type(demo.I).__name__],
     ["parameter",            "c_i",                      "Param",         type(demo.c).__name__],
     ["decision variable",    "x_i >= 0",                 "Var",           type(demo.x).__name__],
     ["inequality constraint","sum_i x_i - 9 <= 0",       "Constraint",    type(demo.cap).__name__],
     ["objective",            "max sum_i c_i x_i",        "Objective",     type(demo.margin).__name__]],
    columns=["algebraic object", "notation", "Pyomo component", "Python class"])

print(mapping.to_string(index=False))
print()
print("number of decision variables n =", len([v for v in demo.component_data_objects(pyo.Var)]))
print("number of constraints        m =", len([c for c in demo.component_data_objects(pyo.Constraint)]))
print("objective sense                =", "maximize" if demo.margin.sense == pyo.maximize else "minimize")

## 2. The optimization workflow: five stages

Section 1 named the parts of a model. It did not say how to get from a paragraph written by a process engineer to a number a manager can act on. That path is the same every time, and it is worth naming, because almost every serious error in applied optimization can be located at one specific stage.

    Stage 1            Stage 2                  Stage 3            Stage 4      Stage 5
    verbal      ->     structured        ->     algebraic    ->    code   ->    solution and
    problem            specification            model                           interpretation

Each stage takes the artifact produced by the previous one and produces a new artifact that is more formal and less ambiguous. The stages are not interchangeable and they are not optional. Skipping stage 2 and writing Pyomo directly from a paragraph is the single most common cause of a model that runs, returns a number, and is wrong.

| Stage | Question it answers | Artifact it produces | Characteristic failure |
|---|---|---|---|
| 1. Verbal problem | What decision is actually being made, and by whom? | A paragraph of prose, plus the data source for every number in it | Optimizing the wrong thing, or a decision the plant cannot execute |
| 2. Structured specification | What are the sets, the data, the decisions, the goal and the restrictions, each with units? | A table: index sets, parameters with units, decision variables with units and bounds, one objective, a numbered list of restrictions in words | Missing constraint, unstated assumption, a parameter treated as a variable |
| 3. Algebraic model | What is the mathematics? | `min f(x)` subject to `g(x) <= 0`, `h(x) = 0`, with every symbol defined and every equation dimensionally consistent | Units that do not balance, an index that appears on one side only, a nonlinearity introduced by accident |
| 4. Code | How is the mathematics expressed to a solver? | A Pyomo `ConcreteModel` with `Set`, `Param`, `Var`, `Constraint`, `Objective`, or a spreadsheet layout | Data typed into model logic, an off-by-one on the index set, a sign error |
| 5. Solution and interpretation | Is the answer correct, and what does it mean? | Termination condition, primal values, duals, a figure, and a sentence an engineer can act on | Reporting numbers without checking the status; quoting a shadow price outside its valid range |

Three rules follow.

- **Stage 2 is written before any algebra.** Its output is a specification in words with units attached. If a quantity has no unit it is not yet understood.
- **Stage 3 is checked dimensionally before stage 4 begins.** A units check on every constraint catches more modeling errors than any other habit, and it costs a minute.
- **Stage 5 begins with the termination condition, not with the objective value.** Section 9 shows two models that return an object full of numbers and no solution at all.

This pipeline is the spine of the modeling block. Weeks 2 to 5 build progressively harder models (linear and multiperiod, networks, blending and pooling, logical and discrete) and every one of them is presented in these five stages. Week 6 uses the same pipeline to audit models drafted by a language model: a generated model is an artifact that appeared at stage 3 or 4 without stages 1 and 2, which is exactly why it must be verified before it is believed.

The rest of this notebook takes one problem through all five stages. Stage 1, in two sentences: *a specialty chemical unit makes an electrolyte additive and a cathode binder on shared equipment, and must choose how much of each to make in a week to earn the largest contribution margin.* Section 4 gives the full statement and the data. The cell below produces the stage 2 artifact for that problem, and Figure 1 draws the pipeline.


In [ ]:
# --- The five-stage workflow, as a table, a specification format and Figure 1 --
WORKFLOW = [
    (1, "verbal problem",         "what decision, by whom",     "prose plus data sources",
        "optimizing the wrong thing"),
    (2, "structured specification","sets, data, decisions, goal, restrictions, units",
        "specification table",        "missing constraint or unstated assumption"),
    (3, "algebraic model",        "what is the mathematics",    "min f(x) s.t. g(x)<=0, h(x)=0",
        "units do not balance"),
    (4, "code",                   "how is it given to a solver","Pyomo ConcreteModel",
        "data buried inside model logic"),
    (5, "solution and interpretation", "is it right, what does it mean",
        "status, values, duals, figure", "numbers reported without the status"),
]
stages = pd.DataFrame(WORKFLOW, columns=["stage", "name", "question", "artifact", "failure"])
with pd.option_context("display.max_colwidth", 52):
    print(stages.to_string(index=False))


def specification(title, sets, params, variables, objective, restrictions):
    """Print the stage 2 artifact: a structured specification with units."""
    print(f"STAGE 2 SPECIFICATION: {title}")
    print("-" * 78)
    for heading, items in [("index sets", sets), ("parameters (data, fixed before solving)", params),
                           ("decision variables", variables)]:
        print(f"{heading}:")
        for sym, meaning, unit in items:
            print(f"   {sym:<12s} {meaning:<44s} [{unit}]")
    print(f"objective:\n   {objective}")
    print("restrictions:")
    for k, r in enumerate(restrictions, start=1):
        print(f"   R{k}. {r}")
    print("-" * 78)


specification(
    title="weekly product mix, specialty chemical unit (stated in Section 4)",
    sets=[("p in P", "products made on the shared equipment", "-"),
          ("r in R", "shared resources with limited weekly hours", "-")],
    params=[("c_p", "contribution margin of product p", "USD/t"),
            ("a_rp", "use of resource r per tonne of product p", "h/t"),
            ("b_r", "hours of resource r available in the week", "h/week")],
    variables=[("x_p >= 0", "tonnes of product p made in the week", "t/week")],
    objective="maximize total weekly contribution margin, sum_p c_p x_p, in USD/week",
    restrictions=["hours used on each resource cannot exceed the hours available",
                  "production cannot be negative",
                  "everything produced can be sold, so there is no demand limit"])

# --- Figure 1: the pipeline ---------------------------------------------------
fig, ax = plt.subplots(figsize=(9.4, 2.0), constrained_layout=True)
labels = ["verbal\nproblem", "structured\nspecification", "algebraic\nmodel", "code",
          "solution and\ninterpretation"]
box_colors = [PALETTE[7], PALETTE[3], PALETTE[0], PALETTE[4], PALETTE[1]]
text_colors = [INK, "white", "white", "white", INK]
for k, (lab, bc, tc) in enumerate(zip(labels, box_colors, text_colors)):
    ax.add_patch(plt.Rectangle((k * 2.0, 0.0), 1.6, 1.0, facecolor=bc,
                               edgecolor=GRAPHITE, lw=1.0, zorder=2))
    ax.text(k * 2.0 + 0.8, 0.5, lab, ha="center", va="center", fontsize=8,
            color=tc, zorder=3)
    ax.text(k * 2.0 + 0.8, 1.16, f"stage {k + 1}", ha="center", va="center",
            fontsize=7.5, color=GRAPHITE, zorder=3)
    if k < len(labels) - 1:
        ax.annotate("", xy=(k * 2.0 + 1.95, 0.5), xytext=(k * 2.0 + 1.62, 0.5),
                    arrowprops=dict(arrowstyle="-|>", color=GRAPHITE, lw=1.6))
ax.set_xlim(-0.15, 9.75); ax.set_ylim(-0.45, 1.45)
ax.axis("off")
ax.text(0.0, -0.30, "engineering judgment", fontsize=7.5, color=GRAPHITE, ha="left")
ax.text(9.6, -0.30, "verification", fontsize=7.5, color=GRAPHITE, ha="right")
ax.set_title("Week 1, Figure 1: the five-stage optimization workflow", fontsize=10)
plt.show()

## 3. The tool stack

Optimization software is organized in two layers, and confusing them is a standing source of trouble.

**The modeling layer** is where you write the model. It knows about sets, parameters, variables, constraints and objectives. It knows nothing about how to solve anything. Excel with Solver or OpenSolver is a modeling layer; so is Pyomo, and so are GAMS, AMPL and JuMP.

**The solver layer** is where the model is solved. It knows about matrices, factorizations, branch and bound trees and convergence tolerances. It knows nothing about your plant. HiGHS, GLPK, CBC, Gurobi, CPLEX, Ipopt, Bonmin, Couenne, SCIP and BARON are solvers.

Between the two sits a thin translation step: the modeling layer writes the model in a standard file format (`.lp`, `.mps` or `.nl`) or passes it through an in-memory API, the solver reads it, solves it, and returns values and status codes that the modeling layer maps back onto your named components.

| Tool | Layer | What it does | Where it stops |
|---|---|---|---|
| Excel Solver | modeling and interface | Target cell, changing cells and constraints on a worksheet; ships with Excel; Simplex LP, GRG Nonlinear and Evolutionary engines | Standard Excel Solver is limited to 200 changing cells and 100 explicit constraints, and the model lives in cell formulas, so it cannot be version controlled or unit tested |
| OpenSolver | modeling and interface | Free Excel add-in that keeps the familiar worksheet layout but sends the model to CBC or HiGHS instead of the built-in engine, removing the size limit | Its default engines are LP and MILP only; nonlinear models still need a different engine or a different tool |
| Pyomo | modeling | Python algebraic modeling language: `Set`, `Param`, `Var`, `Constraint`, `Objective`; separates model structure from data and from solver choice; scales to hundreds of thousands of variables; the same model can be sent to any supported solver | Pyomo solves nothing by itself. Without a solver installed, a Pyomo model is only a data structure |
| Solver layer | solving | Executes the algorithm: simplex or interior point for LP, branch and cut for MILP, interior point or SQP for NLP, branch and bound over NLP relaxations for MINLP | A solver cannot tell you that your model is wrong. It answers the question you asked |

### Solver classes and what each one can and cannot do

The class of the problem, in the sense of Section 8, decides which solvers are even eligible. Sending an MINLP to an LP solver is not slow, it is an error.

| Class | Open source | Commercial | Can handle | Cannot handle | Guarantee |
|---|---|---|---|---|---|
| **LP** linear objective and constraints, continuous | HiGHS, GLPK, CBC (Clp) | Gurobi, CPLEX, XPRESS, MOSEK | Millions of variables; simplex or interior point; duals and sensitivity ranges come free | Any nonlinear term, any integer variable | Global optimum, or a certificate of infeasibility or unboundedness |
| **MILP** linear, some variables integer or binary | HiGHS, CBC, SCIP | Gurobi, CPLEX, XPRESS | Fixed charges, yes-or-no decisions, disjunctions, logical conditions; branch and cut with presolve and cutting planes | General nonlinear terms (some accept convex quadratics) | Global optimum, but worst-case effort grows exponentially; a gap is reported while it runs |
| **NLP** smooth nonlinear, all continuous | Ipopt | CONOPT, Knitro, SNOPT | Kinetics, thermodynamics, bilinear blending, economies of scale; interior point or SQP; needs first and ideally second derivatives | Integer variables, nonsmooth functions such as `abs`, `min` or an if-then rule | A local KKT point. Global only when the problem is convex, and the solver cannot tell you whether it is |
| **MINLP** nonlinear with discrete decisions | Bonmin (convex), Couenne (global), SCIP, MindtPy | BARON, Knitro, ANTIGONE, DICOPT | Unit selection inside a nonlinear flowsheet; process superstructures; branch and bound over NLP relaxations | Large instances; the nonconvex case is the hardest routine class in this course | Bonmin guarantees the optimum only for convex MINLP; Couenne, SCIP and BARON give a global bound at much higher cost |

This course uses Pyomo for the modeling layer throughout, HiGHS, GLPK or CBC for LP and MILP, Ipopt for NLP, and Bonmin or Couenne for MINLP. All are open source. A spreadsheet is used in the Week 1 laboratory only, to make the point that the mathematics does not change with the interface.

The `pick_solver` helper in the environment cell implements exactly the choice above: name a class, receive the first solver of that class that is actually installed. The next cell reports what is installed here and then solves one tiny problem of each class as an end-to-end check of the whole stack.


In [ ]:
# --- Live check: which solvers are installed on this machine ------------------
import logging

CANDIDATES = [
    # (Pyomo name, display name, classes it is used for here, license)
    ("appsi_highs", "HiGHS (appsi interface)", "LP, MILP",     "open source"),
    ("highs",       "HiGHS (direct interface)", "LP, MILP",    "open source"),
    ("glpk",        "GLPK",                   "LP, MILP",      "open source"),
    ("cbc",         "CBC",                    "LP, MILP",      "open source"),
    ("ipopt",       "Ipopt",                  "NLP",           "open source"),
    ("bonmin",      "Bonmin",                 "convex MINLP",  "open source"),
    ("couenne",     "Couenne",                "global MINLP",  "open source"),
    ("scip",        "SCIP",                   "MILP, MINLP",   "academic"),
    ("gurobi",      "Gurobi",                 "LP, MILP, QP",  "commercial"),
    ("cplex",       "CPLEX",                  "LP, MILP, QP",  "commercial"),
    ("baron",       "BARON",                  "global MINLP",  "commercial"),
    ("knitro",      "Knitro",                 "NLP, MINLP",    "commercial"),
]

_log = logging.getLogger("pyomo")
_prev_level = _log.level
_log.setLevel(logging.CRITICAL)          # probing an absent solver is not an error here
try:
    rows = []
    for pname, label, classes, lic in CANDIDATES:
        try:
            s = pyo.SolverFactory(pname)
            ok = bool(s is not None and s.available(exception_flag=False))
        except Exception:
            ok = False
        rows.append({"pyomo name": pname, "solver": label, "used for": classes,
                     "license": lic, "available here": "yes" if ok else "no"})
finally:
    _log.setLevel(_prev_level)

avail_tab = pd.DataFrame(rows)
print("Solver availability on this machine")
print(avail_tab.to_string(index=False))
n_ok = int((avail_tab["available here"] == "yes").sum())
print(f"\n{n_ok} of {len(CANDIDATES)} probed solvers are installed here.")
print("A 'no' is not a fault: commercial solvers need a license, and the course needs none of them.")

In [ ]:
# --- End-to-end smoke test of the stack, one tiny problem per class ----------
# Each model below has a hand-checkable answer, so a wrong install cannot pass silently.
smoke = []

# LP:   max 3a + 2d  s.t. a + d <= 4, a <= 3, a, d >= 0   ->  a = 3, d = 1, z = 11
mlp = pyo.ConcreteModel()
mlp.a = pyo.Var(bounds=(0, 3)); mlp.d = pyo.Var(domain=pyo.NonNegativeReals)
mlp.cap = pyo.Constraint(expr=mlp.a + mlp.d <= 4)
mlp.obj = pyo.Objective(expr=3*mlp.a + 2*mlp.d, sense=pyo.maximize)
r = pick_solver("lp").solve(mlp)
assert r.solver.termination_condition == pyo.TerminationCondition.optimal, "LP smoke test failed"
assert abs(pyo.value(mlp.obj) - 11.0) < 1e-6, "LP smoke test returned the wrong value"
smoke.append(("LP", "max 3a+2d, a+d<=4, a<=3", 11.0, pyo.value(mlp.obj)))

# MILP: same LP with a integer and a + d <= 3.5           ->  a = 3, d = 0.5, z = 10
mip = pyo.ConcreteModel()
mip.a = pyo.Var(bounds=(0, 3), domain=pyo.NonNegativeIntegers)
mip.d = pyo.Var(domain=pyo.NonNegativeReals)
mip.cap = pyo.Constraint(expr=mip.a + mip.d <= 3.5)
mip.obj = pyo.Objective(expr=3*mip.a + 2*mip.d, sense=pyo.maximize)
r = pick_solver("milp").solve(mip)
assert r.solver.termination_condition == pyo.TerminationCondition.optimal, "MILP smoke test failed"
assert abs(pyo.value(mip.obj) - 10.0) < 1e-6, "MILP smoke test returned the wrong value"
smoke.append(("MILP", "same, a integer, a+d<=3.5", 10.0, pyo.value(mip.obj)))

# NLP:  min (a - 2)^2 + (d + 1)^2  s.t. a + d = 0         ->  a = 1.5, d = -1.5, f = 0.5
try:
    mnl = pyo.ConcreteModel()
    mnl.a = pyo.Var(initialize=0.0); mnl.d = pyo.Var(initialize=0.0)
    mnl.eq = pyo.Constraint(expr=mnl.a + mnl.d == 0)
    mnl.obj = pyo.Objective(expr=(mnl.a - 2)**2 + (mnl.d + 1)**2, sense=pyo.minimize)
    r = pick_solver("nlp").solve(mnl)
    assert r.solver.termination_condition == pyo.TerminationCondition.optimal, "NLP smoke test failed"
    assert abs(pyo.value(mnl.obj) - 0.5) < 1e-6, "NLP smoke test returned the wrong value"
    smoke.append(("NLP", "min (a-2)^2+(d+1)^2, a+d=0", 0.5, pyo.value(mnl.obj)))
except RuntimeError as exc:
    smoke.append(("NLP", "min (a-2)^2+(d+1)^2, a+d=0", 0.5, f"skipped: {exc}"))

print(pd.DataFrame(smoke, columns=["class", "tiny problem", "hand answer", "solver answer"])
        .to_string(index=False))
print("\nThe stack is working: the modeling layer, the file interface and the solver layer")
print("all agree with an answer that can be checked by hand.")

## 4. A first model: product mix at a small specialty chemical plant

A small specialty chemical unit supplies the zinc-air battery industry. It makes two products from the same equipment:

- **Additive**, an electrolyte additive that suppresses zinc dendrite growth,
- **Binder**, a binder solution used in air-cathode coating.

Both are made on a shared reactor train, then separated and dried, then finished and packaged. The plant plans one week at a time. Over one week the plant may use

| Resource | Additive, h/t | Binder, h/t | Available, h/week |
|---|---|---|---|
| Reactor train | 3 | 2 | 24 |
| Separation and drying | 1 | 2 | 16 |
| Finishing and packaging | 1 | 1 | 9 |

The reactor train is shared with other campaigns, which is why only 24 hours per week are allocated to these two products. Contribution margin, that is selling price minus variable cost, is 400 USD per tonne of Additive and 300 USD per tonne of Binder. Everything produced can be sold. The plant wants the weekly production plan with the largest total contribution margin.

### Step 1: identify the four parts

**Decision variables.** How much of each product to make in the week:

- `x1 >= 0`, tonnes of Additive per week,
- `x2 >= 0`, tonnes of Binder per week.

Note what is *not* a variable. The margins are not chosen, the hours per tonne are set by the process, and the available hours are set by the schedule. Those are parameters.

**Parameters.** The margins `c = (400, 300)` in USD/t, the resource use matrix `A` in h/t, and the availabilities `b = (24, 16, 9)` in h/week.

**Objective.** Maximize weekly contribution margin, `z = 400 x1 + 300 x2`, in USD/week.

**Constraints.** One per resource, plus non-negativity.

### Step 2: the algebraic model

    maximize    z = 400 x1 + 300 x2

    subject to  3 x1 + 2 x2 <= 24        (reactor train, h/week)
                1 x1 + 2 x2 <= 16        (separation and drying, h/week)
                1 x1 + 1 x2 <=  9        (finishing and packaging, h/week)
                x1 >= 0,  x2 >= 0

In matrix form this is `max c'x` subject to `A x <= b`, `x >= 0`, with

    c = [400, 300]',     A = [[3, 2], [1, 2], [1, 1]],     b = [24, 16, 9]'

Every function here is linear in `x`, and `x` is continuous, so this is a **linear program**. Check the units on every constraint before going further: `(h/t) * (t/week) = h/week`, which matches the right-hand side. A units check catches more modeling errors than any other single habit.

In [ ]:
# --- Data ---------------------------------------------------------------------
PRODUCTS  = ["Additive", "Binder"]
RESOURCES = ["reactor", "separation", "packaging"]

margin = pd.Series({"Additive": 400.0, "Binder": 300.0}, name="USD_per_t")

usage = pd.DataFrame(
    [[3.0, 2.0],
     [1.0, 2.0],
     [1.0, 1.0]],
    index=RESOURCES, columns=PRODUCTS)          # hours per tonne

avail = pd.Series({"reactor": 24.0, "separation": 16.0, "packaging": 9.0},
                  name="h_per_week")

print("Contribution margin, USD per tonne")
print(margin.to_string(), "\n")
print("Resource use, hours per tonne")
print(usage.to_string(), "\n")
print("Resource availability, hours per week")
print(avail.to_string())

# The three arrays that define the LP.
c = margin.values                 # (2,)
A = usage.values                  # (3, 2)
b = avail.values                  # (3,)
print(f"\nc shape {c.shape}, A shape {A.shape}, b shape {b.shape}")
print(f"n = {A.shape[1]} decision variables, m = {A.shape[0]} resource constraints")

## 5. Solution method 1: by hand and graphically

With two decision variables the feasible region is a subset of the plane, and it can be drawn. Two facts make the drawing enough to solve the problem exactly.

1. **The feasible region of an LP is a convex polyhedron.** Each constraint `a'x <= b_i` is a half plane. The intersection of half planes is convex and has flat faces.
2. **If an LP has a finite optimum, at least one optimal solution is at a vertex** (a corner, formally an extreme point). The reason is visible in the picture: the objective contours `c'x = z` are parallel straight lines, and pushing a straight line as far as possible across a polygon in the direction of the gradient stops at a corner, or along a whole edge if the line happens to be parallel to that edge.

So the hand method is: find every vertex, discard the infeasible ones, evaluate `z` at the rest, and take the best. A vertex in two dimensions is the intersection of two constraint boundaries, including the axes `x1 = 0` and `x2 = 0`. With five boundaries there are at most `C(5,2) = 10` intersections to test. This enumeration is exactly what the simplex method automates in Week 7, except that simplex walks from vertex to vertex uphill instead of listing them all.

The objective gradient is `grad z = c = (400, 300)`, which points up and to the right. Isoprofit lines `400 x1 + 300 x2 = z` are perpendicular to it, with slope `-400/300 = -4/3`.

In [ ]:
# --- Vertex enumeration, done explicitly --------------------------------------
# Boundaries, written as a'x = rhs. The last two are the axes.
bounds_A   = np.vstack([A, np.array([[1.0, 0.0], [0.0, 1.0]])])
bounds_rhs = np.concatenate([b, np.array([0.0, 0.0])])
bound_name = RESOURCES + ["x1 = 0", "x2 = 0"]

TOL = 1e-9

def feasible(x):
    """True if x satisfies A x <= b and x >= 0, within tolerance."""
    return bool(np.all(A @ x <= b + 1e-7) and np.all(x >= -1e-7))

rows = []
for i in range(len(bound_name)):
    for j in range(i + 1, len(bound_name)):
        M = bounds_A[[i, j], :]
        if abs(np.linalg.det(M)) < TOL:          # parallel boundaries, no intersection
            continue
        x = np.linalg.solve(M, bounds_rhs[[i, j]])
        rows.append({"boundaries": f"{bound_name[i]} & {bound_name[j]}",
                     "x1": x[0], "x2": x[1],
                     "feasible": feasible(x),
                     "z_USD_per_week": float(c @ x)})

vertices = pd.DataFrame(rows)
vertices.loc[~vertices.feasible, "z_USD_per_week"] = np.nan
print("All pairwise intersections of constraint boundaries")
print(vertices.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))

feas = vertices[vertices.feasible].copy()
best = feas.loc[feas.z_USD_per_week.idxmax()]
x_graph = np.array([best.x1, best.x2])
z_graph = float(best.z_USD_per_week)

print(f"\nFeasible vertices: {len(feas)} of {len(vertices)} intersections")
print(f"Best vertex      : x1 = {x_graph[0]:.4g} t/week, x2 = {x_graph[1]:.4g} t/week")
print(f"Optimal value    : z  = {z_graph:,.2f} USD/week")
print(f"Binding at the optimum: "
      f"{[r for r, s in zip(RESOURCES, A @ x_graph - b) if abs(s) < 1e-7]}")

In [ ]:
# --- Figure 1: feasible region, gradient and isoprofit lines ------------------
x1 = np.linspace(0.0, 10.0, 400)

fig, ax = plt.subplots(figsize=(5.8, 4.4), constrained_layout=True)
tidy(ax)

# Feasible region: sample the plane on a fine grid and shade what satisfies A x <= b.
gx, gy = np.meshgrid(np.linspace(0, 10, 600), np.linspace(0, 10, 600))
inside = np.ones_like(gx, dtype=bool)
for k in range(A.shape[0]):
    inside &= (A[k, 0] * gx + A[k, 1] * gy <= b[k] + 1e-12)
ax.contourf(gx, gy, inside.astype(float), levels=[0.5, 1.5],
            colors=[PALETTE[7]], alpha=0.55)

# Constraint boundaries.
line_colors = [PALETTE[3], PALETTE[4], PALETTE[5]]
for k, (name, col) in enumerate(zip(RESOURCES, line_colors)):
    x2_line = (b[k] - A[k, 0] * x1) / A[k, 1]
    ax.plot(x1, x2_line, color=col, lw=1.8,
            label=f"{name}: {A[k,0]:.0f} x1 + {A[k,1]:.0f} x2 <= {b[k]:.0f}")

# Isoprofit family, teal = reference.
for z in [1200.0, 1800.0, 2400.0]:
    ax.plot(x1, (z - c[0] * x1) / c[1], color=PALETTE[0], lw=1.5, ls=":")
ax.plot([], [], color=PALETTE[0], lw=1.5, ls=":", label="isoprofit lines")

# Optimal isoprofit line and the optimum, amber = the effect of interest.
ax.plot(x1, (z_graph - c[0] * x1) / c[1], color=PALETTE[1], lw=2.0,
        label=f"optimal isoprofit, z = {z_graph:,.0f}")
ax.plot([x_graph[0]], [x_graph[1]], marker="o", ms=8, color=PALETTE[1], zorder=6)
ax.annotate(f"x* = ({x_graph[0]:.0f}, {x_graph[1]:.0f})\nz* = {z_graph:,.0f} USD/week",
            xy=(x_graph[0], x_graph[1]), xytext=(1.6, 1.1),
            fontsize=8, color=INK,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=MIST, linewidth=0.8),
            arrowprops=dict(arrowstyle="->", color=GRAPHITE, lw=1.2))

# Objective gradient, scaled to fit the axes. Graphite, as a reference direction.
g = c / np.linalg.norm(c) * 2.4
ax.annotate("", xy=(x_graph[0] + g[0], x_graph[1] + g[1]), xytext=(x_graph[0], x_graph[1]),
            arrowprops=dict(arrowstyle="-|>", color=GRAPHITE, lw=2.0), zorder=7)
ax.text(x_graph[0] + g[0] + 0.15, x_graph[1] + g[1] + 0.35, "grad z = (400, 300)",
        fontsize=8, color=INK, va="center", ha="right")

# Feasible vertices.
ax.scatter(feas.x1, feas.x2, s=26, facecolor="white", edgecolor=GRAPHITE,
           zorder=4, linewidths=1.1, label="feasible vertices")

ax.set_xlim(-0.35, 10); ax.set_ylim(-0.35, 10)
ax.set_xlabel("x1, Additive, tonnes per week")
ax.set_ylabel("x2, Binder, tonnes per week")
ax.set_title("Week 1, Figure 2: graphical solution of the product-mix LP", fontsize=10)
ax.legend(fontsize=7.2, loc="upper right")
plt.show()

## 6. Solution method 2: the spreadsheet layout

Excel Solver and OpenSolver ask for three things: the **target cell** (the objective), the **changing cells** (the decision variables) and the **constraints** (a left-hand-side cell, a relation, and a right-hand-side cell). A spreadsheet solves the same LP as Pyomo. The only difference is that the model lives in cell formulas instead of in named components, which is convenient for a model this size and unmanageable for a model with a thousand variables.

The layout below is the one to build in the Week 1 laboratory session. Two habits make spreadsheet models auditable: keep data, decisions and formulas in visually separate blocks, and never type a number inside a formula.

In [ ]:
# --- The same LP, laid out as a spreadsheet ----------------------------------
def spreadsheet(x):
    """Return a printable spreadsheet-style layout for a given production plan."""
    lhs = A @ x
    sheet = pd.DataFrame("", index=range(1, 15), columns=list("ABCDEF"))

    sheet.loc[1, "A"] = "PRODUCT MIX, week of 2026-08-17"

    sheet.loc[3, "A"] = "DECISIONS (changing cells B4:C4)"
    sheet.loc[4, ["A", "B", "C"]] = ["tonnes per week", PRODUCTS[0], PRODUCTS[1]]
    sheet.loc[5, ["A", "B", "C"]] = ["", f"{x[0]:.2f}", f"{x[1]:.2f}"]

    sheet.loc[7, "A"] = "OBJECTIVE (target cell D9, maximize)"
    sheet.loc[8, ["A", "B", "C"]] = ["margin, USD/t", f"{c[0]:.0f}", f"{c[1]:.0f}"]
    sheet.loc[9, ["A", "B", "C", "D"]] = ["total margin, USD/week",
                                          "=SUMPRODUCT(B8:C8,B5:C5)", "->",
                                          f"{float(c @ x):,.2f}"]

    sheet.loc[11, ["A", "B", "C", "D", "E", "F"]] = [
        "CONSTRAINTS", "h/t " + PRODUCTS[0], "h/t " + PRODUCTS[1],
        "LHS (=SUMPRODUCT)", "sign", "RHS"]
    for k, r in enumerate(RESOURCES):
        sheet.loc[12 + k, ["A", "B", "C", "D", "E", "F"]] = [
            r, f"{A[k,0]:.0f}", f"{A[k,1]:.0f}", f"{lhs[k]:.2f}", "<=", f"{b[k]:.0f}"]
    return sheet, lhs

sheet, lhs = spreadsheet(x_graph)
print(sheet.to_string())

print("\nSolver dialog, filled in")
print("  Set Objective        : $D$9")
print("  To                   : Max")
print("  By Changing Cells    : $B$5:$C$5")
print("  Subject to           : $D$12:$D$14 <= $F$12:$F$14")
print("                         $B$5:$C$5   >= 0")
print("  Solving Method       : Simplex LP")

slack = b - lhs
print("\nResource check at the graphical optimum")
print(pd.DataFrame({"used_h": lhs, "available_h": b, "slack_h": slack,
                    "binding": np.isclose(slack, 0.0, atol=1e-7)},
                   index=RESOURCES).to_string())

## 7. Solution method 3: Pyomo

The same model once more, this time with each algebraic object given its own named component. Compare it line by line with the algebra in Section 4 and with the mapping table in Section 1, and note that Sections 4 to 7 are stages 2 to 5 of the workflow of Section 2 applied to one problem. The structure is deliberately more elaborate than a two-variable problem needs, because it is the structure that scales: replacing the two products by two hundred requires no change to the model code, only to the data.

Two pieces of practice are introduced here and used in every later notebook.

- **Always check the termination condition.** A solver returns a status as well as numbers. Reading the numbers without reading the status is the most common way to report a wrong answer.
- **Ask for the duals.** `m.dual` collects the shadow price of each constraint, the rate of change of the optimal value per unit increase in the right-hand side. Week 8 develops this properly, but the number is available now and is often the most useful output of the model.

In [ ]:
# --- The product-mix LP in Pyomo ---------------------------------------------
def build_product_mix(avail_h=None, name="product_mix"):
    """Build the product-mix LP as a ConcreteModel. avail_h overrides availability."""
    bb = avail.copy() if avail_h is None else pd.Series(avail_h)

    m = pyo.ConcreteModel(name=name)

    # Sets
    m.P = pyo.Set(initialize=PRODUCTS, doc="products")
    m.R = pyo.Set(initialize=RESOURCES, doc="shared resources")

    # Parameters
    m.margin = pyo.Param(m.P, initialize=margin.to_dict(), doc="USD per tonne")
    m.use    = pyo.Param(m.R, m.P,
                         initialize={(r, p): float(usage.loc[r, p])
                                     for r in RESOURCES for p in PRODUCTS},
                         doc="hours per tonne")
    m.avail  = pyo.Param(m.R, initialize=bb.to_dict(), doc="hours per week")

    # Decision variables
    m.x = pyo.Var(m.P, domain=pyo.NonNegativeReals, doc="tonnes per week")

    # Constraints
    def resource_rule(m, r):
        return sum(m.use[r, p] * m.x[p] for p in m.P) <= m.avail[r]
    m.resource = pyo.Constraint(m.R, rule=resource_rule)

    # Objective
    m.profit = pyo.Objective(expr=sum(m.margin[p] * m.x[p] for p in m.P),
                             sense=pyo.maximize)

    # Shadow prices
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m

lp_solver = pick_solver("lp")
model = build_product_mix()
res = lp_solver.solve(model)

print("solver status              :", res.solver.status)
print("termination condition      :", res.solver.termination_condition)
assert res.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "the product-mix LP did not solve to optimality"

x_pyomo = np.array([pyo.value(model.x[p]) for p in PRODUCTS])
z_pyomo = float(pyo.value(model.profit))
print(f"\noptimal plan               : "
      + ", ".join(f"{p} = {v:.4g} t/week" for p, v in zip(PRODUCTS, x_pyomo)))
print(f"optimal contribution margin: {z_pyomo:,.2f} USD/week")

---

### The same model in the spreadsheet

The companion workbook for this week is `excel/W01_product_mix_STUDENT.xlsx`, and the
completed version is `excel/W01_product_mix_SOLUTION.xlsx`. Per the tool allocation table
of the course specification, Week 1 is an **Excel Solver week**: the spreadsheet is the primary tool and
the Pyomo model above is a mirror of the same LP, so that everyone succeeds on day one without an
installation. The `Model` sheet uses the layout already printed in Section 6.

| Algebraic symbol | Spreadsheet range or layout | Pyomo component | Note |
|---|---|---|---|
| `p in P`, products | column headers `B4:C4` above the `Solution` row | `m.P = pyo.Set(initialize=PRODUCTS)` | The sheet stores an index as a text label; Pyomo stores it as an object that other components can be indexed over. |
| `r in R`, resources | row labels `A12:A14`, one per constraint row | `m.R = pyo.Set(initialize=RESOURCES)` | One spreadsheet row per resource is exactly one `Constraint` index. |
| `c_p`, contribution margin, USD/t | objective coefficient row `B8:C8`, blue text | `m.margin = pyo.Param(m.P, initialize=...)` | Blue means given data: never edited, and never retyped inside a formula. |
| `a_rp`, hours per tonne | coefficient block `B12:C14`, blue text | `m.use = pyo.Param(m.R, m.P, initialize=...)` | The block is the matrix `A`, row `r` by column `p`, in the same orientation as the algebra. |
| `b_r`, hours available per week | `RHS` column `F12:F14`, blue text | `m.avail = pyo.Param(m.R, initialize=...)` | Kept in its own column so that a right-hand side can be changed without editing any formula. |
| `x_p >= 0`, tonnes per week | `Solution` row `B5:C5`, green fill, the changing cells | `m.x = pyo.Var(m.P, domain=pyo.NonNegativeReals)` | Green fill in the sheet and `NonNegativeReals` in Pyomo make the same statement about the same object. |
| non-negativity of `x` | dialog entry `$B$5:$C$5 >= 0`, or the Make Unconstrained Variables Non-Negative box | the `domain` argument of `pyo.Var` | Excel makes non-negativity a checkbox that is easy to forget; Pyomo makes it part of the declaration. |
| `sum_p a_rp x_p <= b_r` | `VALUE` column `D12:D14`, each cell `=SUMPRODUCT($B$5:$C$5,B12:C12)`; relation column `E12:E14` holding `<=`; `RHS` column `F12:F14` | `m.resource = pyo.Constraint(m.R, rule=resource_rule)` | Dialog entry `$D$12:$D$14 <= $F$12:$F$14`. The absolute reference to the `Solution` row is what lets one formula be copied down the block. |
| `z = sum_p c_p x_p`, maximize | objective cell `D9`, `=SUMPRODUCT(B8:C8,$B$5:$C$5)`, entered as Set Objective with To: Max | `m.profit = pyo.Objective(expr=..., sense=pyo.maximize)` | Structurally identical to a constraint row; only its role in the dialog differs. |
| `lambda_r`, shadow price of resource `r` | Sensitivity report, Constraints table, `Shadow Price` column | `m.dual[m.resource[r]]`, enabled by `pyo.Suffix(direction=pyo.Suffix.IMPORT)` | In Excel the report is a checkbox at the end of the solve; in Pyomo the suffix must exist before the solve. |

**Where the spreadsheet stops working.** For this LP it does not: 2 changing cells, 3 constraint rows,
one screen, and Simplex LP returns the same vertex and the same objective value as Pyomo. The limits
appear later in this same notebook. In Section 8 the Additive margin becomes `400 - 20 x1`, so cell `D9`
stops being a `SUMPRODUCT`, the engine must be switched from Simplex LP to GRG Nonlinear, and the answer
now depends on whatever numbers happened to be sitting in `B5:C5` when the dialog was opened. Section 9
asks for a status rather than a number: the infeasible and the unbounded variants produce a message box
that a human has to read and dismiss, where the cells above assert on
`res.solver.termination_condition` and halt the notebook when it is not `optimal`. Even the
finite-difference check of the three shadow prices in Section 7 is three extra solves, each one an edit
to `F12:F14`, a reopened dialog, a click, and a value copied by hand into a scratch cell.

**The same model in four languages.** The appendix notebook `A1_same_model_four_tools.ipynb` writes this
product-mix LP in Excel Solver, Pyomo, JuMP and GAMS side by side, and is the 20 to 30 minute companion
to this section. Its point is that the modeling language is separable from the model.

---

In [ ]:
# --- Results table and shadow prices ------------------------------------------
plan = pd.DataFrame({"tonnes_per_week": x_pyomo,
                     "margin_USD_per_t": margin.values,
                     "contribution_USD": x_pyomo * margin.values},
                    index=PRODUCTS)
plan.loc["TOTAL"] = ["", "", plan.contribution_USD.sum()]
print("Production plan")
print(plan.to_string(), "\n")

used = np.array([sum(usage.loc[r, p] * x_pyomo[PRODUCTS.index(p)] for p in PRODUCTS)
                 for r in RESOURCES])
duals = np.array([model.dual[model.resource[r]] for r in RESOURCES])

resource_table = pd.DataFrame({
    "used_h": used,
    "available_h": avail.values,
    "slack_h": avail.values - used,
    "binding": np.isclose(avail.values - used, 0.0, atol=1e-7),
    "shadow_price_USD_per_h": np.abs(duals),
}, index=RESOURCES)
print("Resource usage and shadow prices")
print(resource_table.to_string(), "\n")

# Independent check of the shadow prices by finite difference.
print("Finite-difference check of each shadow price (add one hour, re-solve)")
for k, r in enumerate(RESOURCES):
    bumped = avail.copy(); bumped[r] += 1.0
    mm = build_product_mix(avail_h=bumped, name=f"bump_{r}")
    rr = lp_solver.solve(mm)
    assert rr.solver.termination_condition == pyo.TerminationCondition.optimal
    delta = float(pyo.value(mm.profit)) - z_pyomo
    print(f"  {r:<11s} predicted {abs(duals[k]):7.2f}   observed {delta:7.2f}   "
          f"{'match' if abs(delta - abs(duals[k])) < 1e-6 else 'MISMATCH'}")

# All three methods must agree.
assert np.allclose(x_graph, x_pyomo, atol=1e-7), "graphical and Pyomo plans differ"
assert abs(z_graph - z_pyomo) < 1e-7, "graphical and Pyomo objective values differ"
assert abs(float(c @ x_pyomo) - z_pyomo) < 1e-7, "spreadsheet arithmetic disagrees"
print(f"\nAll three methods agree: x* = ({x_pyomo[0]:.4g}, {x_pyomo[1]:.4g}), "
      f"z* = {z_pyomo:,.2f} USD/week")

### Interpretation

Read the output above, not the sentences below, if the two ever disagree.

**The three methods return the same plan.** That is the point of the section. The graphical method, the spreadsheet arithmetic and Pyomo are three interfaces to one mathematical object. Later in the course only the third scales, but the geometry in Figure 2 is what you should be seeing in your head when a solver reports an answer.

**The optimum is at a vertex where two constraints bind.** In two dimensions, `n = 2` variables at an optimum are pinned by two active constraints. The reactor train and the finishing line are both fully used; separation and drying is not. This is the general pattern: at a vertex of an LP in `n` variables, `n` constraints are active.

**The slack resource has a zero shadow price.** Buying more separation and drying capacity is worth nothing at this data, because that constraint is not what stops the plan. The two binding resources have positive shadow prices, and those are the numbers to take to a capital request. The finite-difference check re-solves the model with one extra hour of each resource and confirms each shadow price directly, so the values are computed rather than asserted.

**Shadow prices are local.** They are valid only while the same set of constraints stays active. Add enough hours and a different constraint becomes binding, the vertex changes, and the shadow price changes with it. Exercise 3 makes this concrete, and Week 8 gives the exact range over which each value holds.

## 8. Classifying an optimization problem

Classification is not bookkeeping. The class determines which algorithm can solve the problem, how long it will take, and whether the answer you get is a local or a global optimum. Two questions settle almost every case.

1. **Are the objective and all constraints linear in the decision variables?** Products of variables (`x1 x2`), ratios, powers, exponentials, logarithms and trigonometric functions all break linearity.
2. **Must any variable take a discrete value?** Counts of units, yes-or-no decisions and batch multiples all do.

| Answer to 1 | Answer to 2 | Class | Typical solver |
|---|---|---|---|
| linear | all continuous | LP, linear program | simplex, interior point |
| linear | all discrete | IP, integer program | branch and bound |
| linear | mixed | MILP | branch and cut |
| nonlinear | all continuous | NLP | gradient based, SQP, interior point |
| nonlinear | mixed | MINLP | branch and bound with NLP subproblems |

Two further features cut across the table.

- **Uncertainty.** If some parameter is not known when the decision is made and is described by a probability distribution or a scenario set, the problem is **stochastic**. The usual structure is two stage: decide now, observe, then take a recourse action.
- **Nonconvexity.** If the feasible region or the objective is nonconvex, a local optimum need not be global, and finding a certified global optimum requires **global optimization** methods rather than a local NLP solver.

### Six problem statements

Classify each, then compare with the table produced by the next cell.

1. A refinery blends three feedstocks into one gasoline grade. Blending costs are linear in the volumes, and octane, sulfur and vapor pressure limits are linear in the volumes. Choose the volumes to minimize cost.
2. A jacketed CSTR carries an exothermic reaction with Arrhenius kinetics. Choose the operating temperature and the residence time, both continuous, to maximize the yield of the desired product.
3. Eight candidate heat exchanger matches are available. Each is either installed or not, at a known fixed cost and a known fixed duty. Choose the set of matches that meets the duty target at least installed cost.
4. A battery-materials producer may build plants at five candidate sites, each a yes-or-no decision with a fixed capital cost, and must decide how many tonnes to ship on each lane. All costs are linear in the tonnages.
5. Cell production capacity for next quarter must be committed now, before demand is known. Demand is described by three equally likely scenarios, and unmet demand is bought on the spot market at a penalty once the scenario is revealed.
6. An equivalent-circuit model of a zinc-air cell is fitted to voltage data by least squares. The residual surface is nonconvex with many local minima, and the reported fit must be the best one, not merely a good one.

In [ ]:
# --- Classification table ----------------------------------------------------
statements = [
    ("1. Gasoline blending, linear costs and linear specification limits",
     "LP",
     "Objective and all constraints linear; all volumes continuous."),
    ("2. CSTR temperature and residence time with Arrhenius kinetics",
     "NLP",
     "Rate constant k = k0 exp(-E/RT) is nonlinear in T; variables continuous."),
    ("3. Choose which of eight heat exchanger matches to install",
     "IP",
     "Every variable is a yes-or-no decision; costs and duties linear. All discrete."),
    ("4. Site five candidate plants and route tonnages on each lane",
     "MILP",
     "Binary build decisions plus continuous flows, all relations linear."),
    ("5. Commit cell capacity before demand is revealed, three scenarios",
     "stochastic",
     "A parameter is unknown at decision time; recourse follows the observation."),
    ("6. Least-squares fit of an equivalent-circuit model, many local minima",
     "global",
     "Nonconvex residual surface; a local NLP solver cannot certify the best fit."),
]

clf = pd.DataFrame(statements, columns=["problem statement", "class", "deciding feature"])
with pd.option_context("display.max_colwidth", 80):
    print(clf.to_string(index=False))

print("\nWhy the class matters")
for cls, note in [
    ("LP",         "solved reliably at very large scale; the optimum is global"),
    ("NLP",        "a solver returns a local optimum unless the problem is convex"),
    ("IP / MILP",  "worst-case effort grows exponentially; formulation strength matters"),
    ("stochastic", "size grows with the number of scenarios; the answer hedges"),
    ("global",     "certifying a global optimum costs far more than finding a local one"),
]:
    print(f"  {cls:<11s} {note}")

### What nonlinearity and integrality actually do

Two small experiments on the same product-mix problem, so that the effect of each feature is isolated.

**Nonlinearity.** Suppose the Additive cannot be sold in unlimited quantity at 400 USD/t. Pushing volume onto the market depresses the price, so the realized margin is `400 - 20 x1` USD/t and the contribution from Additive is `(400 - 20 x1) x1`. The objective becomes

    maximize    z = 400 x1 - 20 x1^2 + 300 x2

with the same three linear constraints. The objective is concave and the feasible set is convex, so the optimum is still global, but it is no longer forced to a vertex.

**Integrality.** Suppose instead that the reactor can only be charged in whole 4 tonne batches, so `x1 = 4 a` and `x2 = 4 d` with `a` and `d` non-negative integers. The feasible set is no longer a polygon, it is a finite set of lattice points inside the polygon.

In [ ]:
# --- Experiment A: one nonlinear term -----------------------------------------
nlp_solver = pick_solver("nlp")

mn = pyo.ConcreteModel(name="price_depression")
mn.P = pyo.Set(initialize=PRODUCTS)
mn.R = pyo.Set(initialize=RESOURCES)
mn.x = pyo.Var(mn.P, domain=pyo.NonNegativeReals, initialize=1.0)
mn.resource = pyo.Constraint(
    mn.R, rule=lambda m, r: sum(usage.loc[r, p] * m.x[p] for p in PRODUCTS) <= avail[r])
mn.profit = pyo.Objective(
    expr=(400.0 - 20.0 * mn.x["Additive"]) * mn.x["Additive"] + 300.0 * mn.x["Binder"],
    sense=pyo.maximize)

rn = nlp_solver.solve(mn)
assert rn.solver.termination_condition == pyo.TerminationCondition.optimal
x_nlp = np.array([pyo.value(mn.x[p]) for p in PRODUCTS])
z_nlp = float(pyo.value(mn.profit))
print(f"NLP optimum : x = ({x_nlp[0]:.4f}, {x_nlp[1]:.4f}) t/week, "
      f"z = {z_nlp:,.2f} USD/week")
act = [r for r, s in zip(RESOURCES, A @ x_nlp - b) if abs(s) < 1e-5]
print(f"active constraints at the NLP optimum: {act}  "
      f"({len(act)} active, so it is not a vertex)")

# --- Experiment B: integer batches --------------------------------------------
milp_solver = pick_solver("milp")
BATCH = 4.0

mi = pyo.ConcreteModel(name="whole_batches")
mi.P = pyo.Set(initialize=PRODUCTS)
mi.R = pyo.Set(initialize=RESOURCES)
mi.n = pyo.Var(mi.P, domain=pyo.NonNegativeIntegers, doc="number of 4 t batches")
mi.resource = pyo.Constraint(
    mi.R, rule=lambda m, r: sum(usage.loc[r, p] * BATCH * m.n[p] for p in PRODUCTS)
                            <= avail[r])
mi.profit = pyo.Objective(
    expr=sum(margin[p] * BATCH * mi.n[p] for p in PRODUCTS), sense=pyo.maximize)

ri = milp_solver.solve(mi)
assert ri.solver.termination_condition == pyo.TerminationCondition.optimal
x_int = np.array([BATCH * pyo.value(mi.n[p]) for p in PRODUCTS])
x_int = np.where(np.abs(x_int) < 1e-9, 0.0, x_int)      # avoid printing a negative zero
z_int = float(pyo.value(mi.profit))
print(f"\nIP optimum  : x = ({x_int[0]:.0f}, {x_int[1]:.0f}) t/week, "
      f"z = {z_int:,.2f} USD/week")

# What rounding the LP answer down to whole batches would have given.
x_round = np.floor(x_pyomo / BATCH) * BATCH
z_round = float(c @ x_round)
print(f"LP answer rounded down to whole batches: x = ({x_round[0]:.0f}, {x_round[1]:.0f}), "
      f"z = {z_round:,.2f} USD/week")

print(f"\ncontinuous LP    z = {z_pyomo:9,.2f} USD/week   (upper bound on both variants)")
print(f"concave NLP      z = {z_nlp:9,.2f} USD/week   "
      f"(lower: the price falls as volume rises)")
print(f"whole batches    z = {z_int:9,.2f} USD/week   "
      f"(lower: only lattice points are allowed)")
print(f"naive rounding   z = {z_round:9,.2f} USD/week   "
      f"({100 * (z_int - z_round) / z_int:.1f} percent below the true integer optimum)")

In [ ]:
# --- Figure 2: how nonlinearity and integrality move the optimum -------------
fig, ax = plt.subplots(figsize=(5.8, 4.4), constrained_layout=True)
tidy(ax)

ax.contourf(gx, gy, inside.astype(float), levels=[0.5, 1.5],
            colors=[PALETTE[7]], alpha=0.55)
for k, col in enumerate([PALETTE[3], PALETTE[4], PALETTE[5]]):
    ax.plot(x1, (b[k] - A[k, 0] * x1) / A[k, 1], color=col, lw=1.5)

# Lattice of whole-batch plans that are feasible.
lat = [(BATCH * i, BATCH * j)
       for i in range(6) for j in range(6)
       if np.all(A @ np.array([BATCH * i, BATCH * j]) <= b + 1e-9)]
lat = np.array(lat)
ax.scatter(lat[:, 0], lat[:, 1], s=34, marker="s", facecolor="white",
           edgecolor=GRAPHITE, linewidths=1.1, zorder=4,
           label="feasible whole-batch plans")

ax.plot([x_pyomo[0]], [x_pyomo[1]], marker="o", ms=9, color=PALETTE[0], zorder=6,
        label=f"LP optimum, z = {z_pyomo:,.0f}")
ax.plot([x_nlp[0]], [x_nlp[1]], marker="^", ms=9, color=PALETTE[1], zorder=6,
        label=f"NLP optimum, z = {z_nlp:,.0f}")
ax.plot([x_int[0]], [x_int[1]], marker="s", ms=9, color=PALETTE[2], zorder=6,
        label=f"IP optimum, z = {z_int:,.0f}")

ax.set_xlim(-0.35, 10); ax.set_ylim(-0.35, 10)
ax.set_xlabel("x1, Additive, tonnes per week")
ax.set_ylabel("x2, Binder, tonnes per week")
ax.set_title("Week 1, Figure 3: same constraints, three problem classes", fontsize=10)
ax.legend(fontsize=7.2, loc="upper right")
plt.show()

Read Figure 3 with the printed numbers next to it. The LP optimum sits at a corner. The NLP optimum sits on the interior of an edge, because the curved objective stops improving before the corner is reached, which is why vertex enumeration is useless for nonlinear problems. The IP optimum is a lattice point, and it is not the lattice point nearest the LP optimum. That last observation is the practical warning: rounding an LP answer to satisfy integrality is not a method. It can give a badly suboptimal plan, and on a tighter problem it can give an infeasible one.

## 9. Feasibility and boundedness

A solver can return three qualitatively different outcomes, and all three are informative.

- **Optimal.** A feasible solution exists and the objective cannot be improved. Read the numbers.
- **Infeasible.** No `x` satisfies every constraint at once. The feasible region is empty. This is a statement about the model, not about the solver: the constraints as written contradict each other, or the data are wrong.
- **Unbounded.** The feasible region is non-empty but the objective can be improved without limit. In a physical problem this always means a constraint has been forgotten, because no real plant can earn infinite margin.

Both faults are diagnosed here by deliberately breaking the working model.

**Infeasible variant.** Add a supply contract that obliges the plant to deliver at least 12 t/week of Additive and Binder combined: `x1 + x2 >= 12`. The finishing line already caps the total at `x1 + x2 <= 9`. The two requirements cannot both hold.

**Unbounded variant.** Delete all three resource constraints and keep only a product-ratio requirement, that Binder output not exceed Additive output by more than 5 t/week: `x2 - x1 <= 5`. Nothing now limits the scale of production. Increasing `x1` and `x2` together along the ray `x(t) = (t, t)` satisfies that constraint for every `t >= 0` and increases the margin without limit.

In [ ]:
# --- Infeasible variant -------------------------------------------------------
m_inf = build_product_mix(name="infeasible_variant")
m_inf.contract = pyo.Constraint(expr=sum(m_inf.x[p] for p in PRODUCTS) >= 12.0)

r_inf = lp_solver.solve(m_inf, load_solutions=False)
print("INFEASIBLE VARIANT: contract x1 + x2 >= 12 against packaging x1 + x2 <= 9")
print("  solver status        :", r_inf.solver.status)
print("  termination condition:", r_inf.solver.termination_condition)
assert r_inf.solver.termination_condition == pyo.TerminationCondition.infeasible, \
    "the infeasible variant was expected to be reported infeasible"
print("  no solution was loaded into the model, because there is none to load")

# The contradiction, shown arithmetically rather than asserted.
print(f"\n  packaging allows at most {b[2]:.0f} t/week in total "
      f"(1 h/t for either product, {b[2]:.0f} h/week available)")
print(f"  the contract demands at least 12 t/week in total")
print(f"  12 > {b[2]:.0f}, so the feasible region is empty")
print("  Remedies: relax the contract, buy packaging capacity, or toll out the balance.")

In [ ]:
# --- Unbounded variant --------------------------------------------------------
m_unb = pyo.ConcreteModel(name="unbounded_variant")
m_unb.P = pyo.Set(initialize=PRODUCTS)
m_unb.x = pyo.Var(m_unb.P, domain=pyo.NonNegativeReals)
m_unb.ratio = pyo.Constraint(expr=m_unb.x["Binder"] - m_unb.x["Additive"] <= 5.0)
m_unb.profit = pyo.Objective(expr=sum(margin[p] * m_unb.x[p] for p in PRODUCTS),
                             sense=pyo.maximize)

r_unb = lp_solver.solve(m_unb, load_solutions=False)
print("UNBOUNDED VARIANT: only the ratio constraint x2 - x1 <= 5 is kept")
print("  solver status        :", r_unb.solver.status)
print("  termination condition:", r_unb.solver.termination_condition)
assert r_unb.solver.termination_condition in (pyo.TerminationCondition.unbounded,
                                              pyo.TerminationCondition.infeasibleOrUnbounded), \
    "the unbounded variant was expected to be reported unbounded"

# Exhibit the improving ray explicitly: x(t) = (t, t) stays feasible for every t >= 0.
print("\n  improving ray x(t) = (t, t), which satisfies x2 - x1 = 0 <= 5 for every t >= 0")
for t in [10.0, 100.0, 1000.0, 10000.0]:
    xt = np.array([t, t])
    print(f"    t = {t:8.0f}   feasible: {bool(xt[1] - xt[0] <= 5.0)}   "
          f"z = {float(c @ xt):12,.0f} USD/week")
print("  z grows without limit along this ray, so no optimal solution exists.")
print("  Remedy: restore the resource constraints that were deleted. An unbounded")
print("  model is a missing constraint, never a solver defect.")

### Reading the status

Three habits follow from the two experiments.

**Assert the termination condition after every solve, not the solver status.** Notice in the output above that the coarse `solver.status` field reads `error` for both broken variants, which says only that no solution was produced. The informative field is `solver.termination_condition`, which distinguishes `infeasible` from `unbounded`. Both variants return an object with numbers in it if you do not check. The `assert` statements in this notebook are not decoration: they are the reason a wrong answer cannot pass silently.

**Diagnose infeasibility on the constraints, not on the solver.** The productive question is which subset of constraints is mutually contradictory. Here the contradiction is visible by arithmetic, `12 > 9`. In a large model, drop constraints in groups until the model becomes feasible, or ask the solver for an irreducible infeasible subset if it supports one. Adding an elastic slack variable with a large penalty is also common: the model then always solves, and the slack values point at the binding contradiction.

**Treat unboundedness as a modeling error.** The improving ray printed above is the direction along which the objective grows forever. Find the ray, and the missing constraint is usually obvious: here, deleting the reactor and packaging limits removed everything that stopped production from growing.

## 10. Exercises

**Exercise 1 (introductory).** A third product, a Separator coating solution, has a contribution margin of 260 USD/t and needs 1 h/t of reactor, 1 h/t of separation and drying, and 2 h/t of finishing and packaging. Add it to the Pyomo model as a third member of the set `P`, re-solve, and report the new plan and the new total margin. Explain in one sentence, using the shadow prices printed in Section 7, why the new product does or does not enter the plan.

**Exercise 2 (introductory).** Take the base model and reduce the finishing and packaging availability from 9 to 7 h/week. Predict the new optimal margin using the shadow price from Section 7 before you solve, then solve and compare. Redraw Figure 2 with the new constraint line and identify the new optimal vertex by name.

**Exercise 3 (intermediate).** Sweep the reactor availability from 18 to 32 h/week in steps of 0.5 h and plot the optimal margin against it, using the palette. The curve is piecewise linear. Report the reactor hours at which the slope changes and explain which constraint becomes binding at that point. This demonstrates directly why a shadow price is valid only over a range.

**Exercise 4 (intermediate).** Return to the nonlinear variant of Section 8, in which the Additive margin is `400 - 20 x1`. Repeat the solve from at least ten different starting points spread over the feasible region and confirm that every run reaches the same optimum. Then replace the concave term by the nonconvex objective `z = 400 x1 - 20 x1^2 + 300 x2 + 30 x1 x2`, repeat the multistart, and report how many distinct local optima you find. Explain what changed.

**Exercise 5 (advanced).** The plant may rent extra reactor hours at 60 USD/h, up to 8 h/week, and extra finishing hours at 130 USD/h, up to 4 h/week. Add rental variables to the model, re-solve, and report how many hours of each are rented and the new margin net of rental cost. Then verify your answer against the shadow prices of the base model: explain precisely why the base shadow prices predict the direction of the answer correctly but do not predict the amount rented, and state what would have to be true for them to predict it exactly.

**Exercise 6 (introductory, cross-tool).** Build the `Model` sheet of
`excel/W01_product_mix_STUDENT.xlsx` exactly as laid out in Section 6: `Solution` row
`B5:C5`, objective coefficient row `B8:C8`, objective cell `D9`, and the three constraint rows with
their `VALUE`, relation and `RHS` columns. Run Solver with the Simplex LP engine and request the
sensitivity report. Confirm three agreements with the Pyomo run of Section 7: cell `D9` equals the
optimal contribution margin to the cent, `B5:C5` equals the optimal plan to four significant figures,
and the `Shadow Price` column of the sensitivity report equals the shadow prices printed in the resource
table, resource by resource. Report any disagreement together with the cell that caused it, and state
which of the two representations makes that error easier to find.


## 11. Takeaways

- Every optimization problem is four things: decision variables, parameters, one scalar objective, and constraints. Identify all four in words before writing any algebra, and check the units of every constraint. A quantity you cannot change is a parameter, not a variable.
- Pyomo has one named component for each algebraic object: `Set` for indices, `Param` for data, `Var` for decisions, `Constraint` for restrictions, `Objective` for the goal. Writing the algebra first makes the code mechanical, and keeping data separate from model logic is what lets the same code serve two products or two hundred.
- A graphical solution, a spreadsheet and a Pyomo model are three interfaces to the same mathematical object and must return the same answer. Cross-checking them is the cheapest verification available, and it is the habit this course expects on every result.
- The optimum of a linear program with a finite optimal value lies at a vertex of the feasible region, where `n` constraints are active. That fact is the basis of the simplex method in Week 7. It fails as soon as the objective becomes nonlinear, and lattice points replace the polygon as soon as a variable becomes integer.
- Classification decides the algorithm and the strength of the guarantee. Linear and continuous gives a global optimum reliably; nonlinear and nonconvex gives a local one; integrality makes the effort grow exponentially; unknown parameters call for a stochastic formulation. Rounding an LP answer is not a substitute for an integer model.
- The solver status is part of the answer. Assert the termination condition after every solve. Infeasible means the constraints contradict each other, unbounded means a constraint is missing, and neither is a defect of the solver.